<img style="float: center;" src='https://github.com/spacetelescope/jwst-pipeline-notebooks/raw/main/_static/stsci_header.png' alt="stsci_logo" width="900px"/>

# Exoplanet Time-Series Observations with MIRI
## From Pixels to Transit/Eclipse Light Curves to Results
### MIRI Photometry Setup Notebook

**Authors**: Taylor James Bell (ESA/AURA for STScI)<br>
**Last Updated**: May 1, 2025<br>
**Pipeline Version**: 1.18.0 (Build 11.3)

**Purpose**:<br>
This setup notebook prepares the necessary files for JWebbinar #41. Data will be located in one observation folder according to path set up below.
It should not be necessary to edit any cells other than in the [Configuration](#1.-Configuration) section.

**Data**:<br>
This notebook is set up to use an example dataset is from [Program ID](https://www.stsci.edu/jwst/science-execution/program-information) 1177 (PI: Greene, Thomas) which is a JWST GTO program.

**JWST pipeline version and CRDS context**:<br>
This notebook was written for the calibration pipeline version given above and uses the context associated with this version of the JWST Calibration Pipeline. Information about this and other contexts can be found in the JWST Calibration Reference Data System (CRDS) [server]((https://jwst-crds.stsci.edu/)). If you use different pipeline
versions, please refer to the table [here](https://jwst-crds.stsci.edu/display_build_contexts/) to determine what context to use. To learn more about the differences for the pipeline, read the relevant [documentation](https://jwst-docs.stsci.edu/jwst-science-calibration-pipeline/jwst-operations-pipeline-build-information)<br>

<hr style="border:1px solid gray"> </hr>

## Table of Contents
* [0. Setting up python environment](#0.-Setting-up-python-environment)
* [1. Configuration](#1.-Configuration)
* [2. Querying MAST using astroquery.mast](#2.-Querying-MAST-using-astroquery.mast)
* [3. Downloading _uncal files for local processing](#3.-Downloading-_uncal-files-for-local-processing)
* [4. Accessing _uncal files from AWS](#4.-Accessing-_uncal-files-from-AWS)

<hr style="border:1px solid gray"> </hr>

## 0. Setting up python environment

The notebooks for JWebbinar #41 should ideally be run in a fresh conda environment to ensure correct installation of all necessary packages. Conda installation instructions can be found [here](https://www.anaconda.com/docs/getting-started/miniconda/install), and once conda is installed you can make a new conda environment for these notebooks by doing

```
conda create -n jwebbinar41 python==3.13.0 nb_conda_kernels -y
```

You can then activate that conda environment and install all the necessary requirements by doing:

```
conda activate jwebbinar41
pip install 'eureka[jwst]@git+https://github.com/kevin218/Eureka.git@v1.2'
```

which will install version 1.2 of Eureka! which includes version 1.18.0 of the jwst pipeline and many other important installation requirements.

The rest of this notebook and all other MIRI/LRS notebooks will assume you are using a conda environment setup as described above.

<hr style="border:1px solid gray"> </hr>

## 1. Configuration

In [1]:
import os
import numpy as np
from astroquery.mast import Observations

In [2]:
path_to_data_folder_on_your_machine = '~/Data/JWST/jwebbinar41/miri_photometry/' # Please adjust to your specific system to specify where you want to download data!

proposal_id = 1177  # This is the program ID for the JWST Transiting Exoplanet Community ERS program
observation_id = 11  # This is the observation ID for the MIRI/LRS phase curve of WASP-43b
visit_id = 1  # There was only one visit as a part of this observation

# For our purposes, we're only going to grab the uncal files
calib_level = 1  # (0 = raw, 1 = uncalibrated, 2 = calibrated, 3 = science product, 4 = contributed science product).
# FITS file type, varies by calib_level.
# 1: UNCAL, GS-ACQ1, GS-ACQ2, GS-FG, GS-ID, GS-TRACK
# 2: CAL, CALINTS, RATE, RATEINTS, X1DINTS, ANNNN_CRFINTS,
# GS-ACQ1, GS-ACQ2, GS-FG, GS-ID, GS-TRACK, RAMP
# 3: X1DINTS, WHTLT
subgroup = 'UNCAL'

<hr style="border:1px solid gray"> </hr>

## 2. Querying MAST using astroquery.mast

In [3]:
# The following code snippet will make sure that the path specified by path_to_data_folder_on_your_machine exists
path_to_data_folder_on_your_machine = os.path.expanduser(path_to_data_folder_on_your_machine)
if not os.path.exists(path_to_data_folder_on_your_machine):
    os.makedirs(path_to_data_folder_on_your_machine, exist_ok=True)

# The following code snippet will make a new directory within path_to_data_folder_on_your_machine called Uncalibrated which will store our downloaded _uncal files
uncaldir = os.path.join(path_to_data_folder_on_your_machine, 'Uncalibrated')
if not os.path.exists(uncaldir):
    os.makedirs(uncaldir, exist_ok=True)

In [4]:
# The following code will convert your inputs to properly formatted strings
if type(proposal_id) is not str:
    proposal_id = str(proposal_id).zfill(5)
if type(observation_id) is not str:
    observation_id = str(observation_id).zfill(3)
if type(visit_id) is not str:
    visit_id = str(visit_id).zfill(3)
if type(calib_level) is int:
    calib_level = [calib_level]

# This code will specify the obsid using wildcards. Note that obs_id comes in two flavors
obs_id = f'jw{proposal_id}{observation_id}{visit_id}_03101*'

# Query MAST for requested visit
sci_table = Observations.query_criteria(proposal_id=proposal_id,
                                        obs_id=obs_id)
table = []
if len(sci_table) > 0:
    # Get product list
    data_products_by_id = Observations.get_product_list(sci_table)
    # Filter for desired files
    table = Observations.filter_products(
        data_products_by_id, productSubGroupDescription=subgroup,
        calib_level=calib_level)

<hr style="border:1px solid gray"> </hr>

## 3. Downloading _uncal files for local processing

The code cell below downloads the _uncal files from MAST for local processing of the data

In [5]:
manifest = Observations.download_products(table, download_dir=uncaldir, flat=True)

<br/>

### 3.1 Checking that the files downloaded successfully
Let's double-check that we have all 6 of the segments we want downloaded and stored in the 'Uncalibrated' folder.

In [6]:
np.sort(os.listdir(uncaldir))

array(['jw01177011001_03101_00001-seg001_mirimage_uncal.fits',
       'jw01177011001_03101_00001-seg002_mirimage_uncal.fits',
       'jw01177011001_03101_00001-seg003_mirimage_uncal.fits',
       'jw01177011001_03101_00001-seg004_mirimage_uncal.fits',
       'jw01177011001_03101_00001-seg005_mirimage_uncal.fits',
       'jw01177011001_03101_00001-seg006_mirimage_uncal.fits',
       'jw01177011001_03101_00001-seg007_mirimage_uncal.fits',
       'jw01177011001_03101_00001-seg008_mirimage_uncal.fits',
       'jw01177011001_03101_00001-seg009_mirimage_uncal.fits',
       'jw01177011001_03101_00001-seg010_mirimage_uncal.fits'],
      dtype='<U52')

<hr style="border:1px solid gray"> </hr>

## 4. Accessing _uncal files from AWS

If you are working on AWS, you can access the _uncal FITS files directly without having to first download them. To do this, you can find the paths to the FITS files using the following cells

In [7]:
# If you don't already have boto3 installed, you will need to install it by uncommenting the line below

!pip install boto3

  Using cached boto3-1.38.7-py3-none-any.whl.metadata (6.6 kB)
Using cached boto3-1.38.7-py3-none-any.whl (139 kB)


In [8]:
Observations.enable_cloud_dataset()
Observations.get_cloud_uris(table)

INFO: Using the S3 STScI public dataset [astroquery.mast.cloud]


['s3://stpubdata/jwst/public/jw01177/jw01177011001/jw01177011001_03101_00001-seg001_mirimage_uncal.fits',
 's3://stpubdata/jwst/public/jw01177/jw01177011001/jw01177011001_03101_00001-seg002_mirimage_uncal.fits',
 's3://stpubdata/jwst/public/jw01177/jw01177011001/jw01177011001_03101_00001-seg003_mirimage_uncal.fits',
 's3://stpubdata/jwst/public/jw01177/jw01177011001/jw01177011001_03101_00001-seg004_mirimage_uncal.fits',
 's3://stpubdata/jwst/public/jw01177/jw01177011001/jw01177011001_03101_00001-seg005_mirimage_uncal.fits',
 's3://stpubdata/jwst/public/jw01177/jw01177011001/jw01177011001_03101_00001-seg006_mirimage_uncal.fits',
 's3://stpubdata/jwst/public/jw01177/jw01177011001/jw01177011001_03101_00001-seg007_mirimage_uncal.fits',
 's3://stpubdata/jwst/public/jw01177/jw01177011001/jw01177011001_03101_00001-seg008_mirimage_uncal.fits',
 's3://stpubdata/jwst/public/jw01177/jw01177011001/jw01177011001_03101_00001-seg009_mirimage_uncal.fits',
 's3://stpubdata/jwst/public/jw01177/jw0117701

<hr style="border:1px solid gray"> </hr>

<img style="float: center;" src="https://github.com/spacetelescope/jwst-pipeline-notebooks/raw/main/_static/stsci_footer.png" alt="stsci_logo" width="200px"/> 